# Treinando um modelo de Processamento de Linguagem Natural (NLP) utilizando Aprendizgem por Reforço -> Parte 1

Antes de tudo é nescessário fazer alguns comentários sobre o repositório que estamos utilizando:

- Este repositório é um fork **modificado** do repositório **nlp_gym** que implmenta um ambiente para treinamento de modelos de NLP utilizando Aprendizagem por reforço, através de um ambiente baseado e integrado na biblioteca Gym.

- Para o uso neste projeto algumas partes da estrutura do nlp_gym tiveram que ser modificadas, para resolver problemas de bibliotecas e funções deprecadas, assim como para ficarem compatíveis com os formatos mais recentes do Gym e da biblioteca Stable Baselines.

## Introdução ao Modelo de Aprendizagem por Reforço

A **aprendizagem por reforço** (Reinforcement Learning, RL) é um dos paradigmas centrais de *machine learning* (ML), ao lado de **aprendizagem supervisionada** e **não supervisionada**. Diferente desses, o RL envolve **um agente** que aprende a tomar decisões através da interação com **um ambiente**.

Nesse modelo, o agente:
- **Observa** o estado atual do ambiente.
- **Executa uma ação**.
- **Recebe uma recompensa** (ou penalidade) e uma nova observação do ambiente.

O objetivo principal do agente é **maximizar a soma total das recompensas ao longo do tempo**, aprendendo uma **política ótima** — isto é, uma estratégia que indique qual ação tomar em cada estado.

### Elementos principais
Os componentes básicos do RL incluem:
- **Agente**: o tomador de decisão.
- **Ambiente**: o sistema com o qual o agente interage.
- **Estado (state)**: a representação atual do ambiente.
- **Ação (action)**: o conjunto de decisões disponíveis ao agente.
- **Recompensa (reward)**: o feedback recebido após uma ação.
- **Política (policy)**: a estratégia do agente para escolher ações.

O RL combina conceitos de **exploration** (tentar novas ações) e **exploytation** (aproveitar ações já conhecidas).

## Um loop básico de Aprendizagem por Reforço utilizando Q-Learning (ainda não em NLP)

Para este exemplo utilizaremos um dos algorítimos mais aclassicos de Aprendizagem por reforço, o **Q-Learning**.

**Q-Learning** é um algoritmo popular de aprendizagem por reforço (*Reinforcement Learning*) usado para treinar agentes a encontrar políticas ótimas em ambientes modelados como **Processos de Decisão de Markov (MDP)**.

Ele se baseia na ideia de manter uma **Q-table**, que associa pares de **estado** e **ação** a um valor \( Q(s, a) \), indicando a expectativa de recompensa futura ao tomar a ação \( a \) no estado \( s \).

### Como funciona?

O processo básico segue estes passos:
1. Inicializa a Q-table com valores (geralmente zeros).
2. Em cada episódio:
   - O agente observa o estado atual.
   - Escolhe uma ação usando uma **política** (como *epsilon-greedy*, que mistura exploração e exploração).
   - Executa a ação no ambiente e recebe:
     - Uma nova observação (novo estado),
     - Uma recompensa,
     - Informação se o episódio terminou.
   - Atualiza a Q-table usando a equação de atualização:
   
  $$Q(s, a) \leftarrow Q(s, a) + \alpha \left[ r + \gamma \max_{a'} Q(s', a') - Q(s, a) \right]$$

Onde:
  - $\alpha$: taxa de aprendizado (quanto valor novo influencia o antigo),
  - $\gamma$: fator de desconto (importância das recompensas futuras),
  - $r$: recompensa recebida,
  - $s'$: novo estado,
  - $a'$: próxima ação.

### Relação com o código

No código mostrado:
- A função `base_loop` executa os episódios e passos do agente no ambiente.
- `Q` é a Q-table, atualizada a cada passo usando `update_Q_table`.
- `policy` define como escolher ações (exploração/exploração).
- `epsilon`, `alpha`, `gamma` controlam respectivamente: exploração, aprendizado, e desconto das recompensas.


Ele é um dos métodos base que inspirou algoritmos mais avançados, como **Deep Q-Networks (DQN)**, o qual veremos mais adiante.


## Loop principal do código

In [ ]:
import gymnasium as gym  # Importa a biblioteca Gymnasium, usada para criar ambientes de RL

def base_loop(init_Qtable, ep_num, time_steps, policy, update_Q_table, alpha, gamma, epsilon_min, epsilon_decay):    

    # Cria o ambiente de aprendizado por reforço (ex.: jogos, simulações, etc.)
    env = gym.make("ambiente", render_mode=None)

    # Inicializa a Q-table, que armazena os valores Q para cada estado-ação
    Q = init_Qtable(env)

    # Loop principal: percorre todos os episódios de treinamento
    for ep in range(ep_num):
        acc_reward = 0  # Acumula a recompensa total do episódio
        observation, info = env.reset()  # Reinicia o ambiente para começar um novo episódio
        
        for _ in range(time_steps):  # Para cada passo dentro do episódio (limite de tempo)
            
            # Seleciona uma ação com base na política (ex.: ε-greedy), usando a Q-table e o estado atual
            action = policy(Q, observation, epsilon)

            # Executa a ação no ambiente e obtém o novo estado, recompensa e status do episódio
            new_observation, reward, terminated, truncated, info = env.step(action)
            acc_reward += reward  # Atualiza a recompensa acumulada

            # Se o episódio foi interrompido por limite de tempo, avisa
            if truncated:
                print("episodio truncado")

            # Atualiza a Q-table aplicando a função de atualização (ex.: Q-learning)
            Q = update_Q_table(Q, observation, action, reward, new_observation, alpha, gamma)

            # Atualiza o estado atual para o próximo passo
            observation = new_observation
            
            # Atualiza o valor de epsilon (fator de exploração), reduzindo gradualmente
            epsilon = max(epsilon_min, epsilon * epsilon_decay)

            # Se o episódio terminou (objetivo alcançado ou falha), sai do loop interno
            if terminated or truncated:
                break

## Funções auxiliares

In [ ]:
import numpy as np

def init_Qtable(env):
    """
    Inicializa a Q-table como uma matriz de zeros.
    Dimensão: número de estados x número de ações.
    """
    state_size = env.observation_space.n
    action_size = env.action_space.n
    Q = np.zeros((state_size, action_size))
    return Q

def policy(Q, state, epsilon):
    """
    Política ε-greedy: com probabilidade epsilon, explora (ação aleatória);
    caso contrário, escolhe a ação com maior valor Q.
    """
    if np.random.uniform(0, 1) < epsilon:
        # Exploração: escolhe ação aleatória
        action = np.random.choice(Q.shape[1])
    else:
        # Exploração: escolhe a ação com maior valor Q no estado atual
        action = np.argmax(Q[state, :])
    return action

def update_Q_table(Q, state, action, reward, next_state, alpha, gamma):
    """
    Atualiza a Q-table usando a equação de Q-learning.
    """
    max_future_q = np.max(Q[next_state, :])
    current_q = Q[state, action]
    
    # Equação de atualização:
    # Q(s, a) ← Q(s, a) + α [r + γ * max_a' Q(s', a') - Q(s, a)]
    new_q = current_q + alpha * (reward + gamma * max_future_q - current_q)
    
    Q[state, action] = new_q
    return Q


## O Problema com Q-tables em NLP com Embeddings

### 1. **Espaço de estados contínuo e de alta dimensionalidade**

Em ambientes tradicionais (como `FrozenLake` ou `Taxi-v3` da biblioteca Gymnasium), os **estados são discretos e limitados**, o que torna viável representar a função Q com uma tabela `Q[state, action]`.

Mas em NLP:

* Os **estados** geralmente são **vetores contínuos**, como *embeddings* de palavras, frases ou sentenças.
* Cada vetor pode ter **centenas ou milhares de dimensões**.
* O número total de possíveis vetores (estados) é **praticamente infinito** (ou inumeravelmente grande).

**Resultado:** não é possível indexar uma Q-table por um vetor contínuo de 768 dimensões — ela se tornaria impraticável em termos de memória e generalização.

---

### 2. **Generalização ruim com tabelas**

Q-tables exigem que cada estado seja explicitamente visitado e armazenado.

* Com embeddings, é altamente improvável que o mesmo vetor de entrada ocorra mais de uma vez de forma exata.
* Isso torna impossível a **reutilização** de aprendizados anteriores para estados “parecidos”.

---

### 3. **Explosão de memória**

Suponha que você discretize cada dimensão de um vetor de 300D em apenas 10 valores possíveis. Isso já resulta em:

$$
10^{300} \text{ estados possíveis}
$$

→ Isso é completamente inviável em termos de memória ou armazenamento.

---

## Solução moderna: Deep Q-Networks (DQN)

Para tratar esse problema, a comunidade de RL evoluiu para usar **redes neurais profundas (DNNs)** como aproximadores de função Q, no lugar de tabelas:

### DQN:

* A rede neural recebe o **vetor de embedding** como entrada (um estado contínuo).
* Ela aprende a **aproximar os valores Q(s, a)** diretamente.
* Pode generalizar melhor entre estados semelhantes.
* Permite treinar em domínios como NLP, visão computacional e robótica.

---

## Exemplo em NLP

Um agente de RL que aprende a construir frases, selecionar respostas ou extrair informações pode representar o estado como:

* Um **embedding de sentença**,
* Ou uma concatenação de embeddings (ex: contexto + ação anterior),
* Com ações sendo tokens, palavras ou comandos.

A Q-table não é adequada aqui — usamos redes para **mapear embedding → Q-values**.

### Próxima parte -> Introdução às Redes Neurais